#logisticregression_titanic

In [64]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import accuracy_score,classification_report
from sklearn.impute import SimpleImputer


#LOAD DATASET

In [49]:
df=pd.read_csv('/content/drive/MyDrive/MSC/ML_ALL_PROGRAM_DS_SEM_2/18_Mar/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [50]:
#drop the values
df=df.drop(['PassengerId','Ticket'],axis=1)
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,NaN,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C85,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,NaN,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,C123,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,NaN,S


#Missing Values


In [51]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Cabin,687
Embarked,2


In [52]:
impiter = SimpleImputer(strategy='most_frequent')
df['Age'] = impiter.fit_transform(df[['Age']]).ravel()
df['Embarked'] = impiter.fit_transform(df[['Embarked']]).ravel()

df = df.drop('Cabin', axis=1)
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S


In [53]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


#Feature Engineering

In [54]:
#in this we can remove the Name but Extract Only the It's tital

df['Title']=df['Name'].apply(lambda x:x.split(',')[1].split('.')[0].strip())
df=df.drop('Name',axis=1)
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,0,3,male,22.0,1,0,7.2500,S,Mr
1,1,1,female,38.0,1,0,71.2833,C,Mrs
2,1,3,female,26.0,0,0,7.9250,S,Miss
3,1,1,female,35.0,1,0,53.1000,S,Mrs
4,0,3,male,35.0,0,0,8.0500,S,Mr


#Encoding

In [55]:
le=LabelEncoder()
for col in ['Sex','Embarked','Title']:
  df[col]=le.fit_transform(df[col])

In [56]:
# Split Features And Target

X=df.drop('Survived',axis=1)
y=df['Survived']

In [57]:
# Train test
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


#Random Forest For Feature Importance

In [58]:
rf_temp=RandomForestClassifier(n_estimators=100,random_state=42)
rf_temp.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [59]:
importance=pd.DataFrame({
    'Features':X_train.columns,
    'importance':rf_temp.feature_importances_

})

#Feature Importance

In [60]:
print("\nAll Feature Importance Scores:")
print(importance)


All Feature Importance Scores:
   Features  importance
0    Pclass    0.079145
1       Sex    0.202806
2       Age    0.223758
3     SibSp    0.053510
4     Parch    0.036086
5      Fare    0.257695
6  Embarked    0.034214
7     Title    0.112785


In [61]:

low_features = importance[importance['importance'] < 0.05]['Features']
print("\nDropping Low Importance Features:", list(low_features))

X = X.drop(columns=low_features)



Dropping Low Importance Features: ['Parch', 'Embarked']


In [62]:
#Re Split After Dropping
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [63]:
#Feature Scaling
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


#Decions Forest Model

In [65]:
dt_model=DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train,y_train)

DecisionTreeClassifier(random_state=42)

In [66]:
y_pred_dt=dt_model.predict(X_test)

In [68]:
print("Decrion Tree Result:")

print("Accuracy",accuracy_score(y_test,y_pred_dt))
print("Classification Report:")
print(classification_report(y_test,y_pred_dt))

Decrion Tree Result:
Accuracy 0.7877094972067039
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.78      0.81       105
           1       0.72      0.80      0.76        74

    accuracy                           0.79       179
   macro avg       0.78      0.79      0.78       179
weighted avg       0.79      0.79      0.79       179



# Random Forest Model

In [69]:
rf_model=RandomForestClassifier(n_estimators=100,random_state=4)
rf_model.fit(X_train,y_train)

RandomForestClassifier(random_state=4)

In [71]:
y_pred_rf=rf_model.predict(X_test)
print("Random Forest Result:")

print("Accuracy",accuracy_score(y_test,y_pred_rf))
print("Classification Report:")
print(classification_report(y_test,y_pred_rf))

Random Forest Result:
Accuracy 0.8044692737430168
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.83      0.83       105
           1       0.76      0.77      0.77        74

    accuracy                           0.80       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.80      0.80      0.80       179



#SVM USIng

In [72]:
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)


SVC(kernel='linear')

In [73]:
y_pred_svm=svm_model.predict(X_test)
print("SVM Result:")

print("Accuracy",accuracy_score(y_test,y_pred_svm))
print("Classification Report:")
print(classification_report(y_test,y_pred_svm))


SVM Result:
Accuracy 0.7821229050279329
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.84      0.82       105
           1       0.75      0.70      0.73        74

    accuracy                           0.78       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179



#LogisticRegression

In [74]:
lr_model=LogisticRegression()
lr_model.fit(X_train,y_train)

LogisticRegression()

In [75]:
lr_predict=lr_model.predict(X_test)
print("Logistic Regression ResultL")
print("Accuracy:",accuracy_score(y_test,lr_predict))
print("Classification Report:")
print(classification_report(y_test,lr_predict))

Logistic Regression ResultL
Accuracy: 0.7988826815642458
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.86      0.83       105
           1       0.78      0.72      0.75        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [76]:
#Accuracy Of All For Model
print("Decrion Tree Result:")
print("Accuracy",accuracy_score(y_test,y_pred_dt))
print("Random Forest Result:")
print("Accuracy",accuracy_score(y_test,y_pred_rf))
print("SVM Result:")
print("Accuracy",accuracy_score(y_test,y_pred_svm))
print("Logistic Regression ResultL")
print("Accuracy:",accuracy_score(y_test,lr_predict))

Decrion Tree Result:
Accuracy 0.7877094972067039
Random Forest Result:
Accuracy 0.8044692737430168
SVM Result:
Accuracy 0.7821229050279329
Logistic Regression ResultL
Accuracy: 0.7988826815642458
